# WSL — ranking and VLM supervision
**Prepare/read first. This notebook has not been submitted or run by Codex.**

Launch account: **satyapaladuguplus**. Both notebooks execute in managed **Kaggle Linux**; WSL/Windows identify the launch account, not the cloud operating system.

GPU0:10 supervised D methods. GPU1:Qwen candidate/prefix/replay labels,7 P students,2 feedback RL methods, each gated.

**Single trial:** seed1729, at most50 supervised epochs with patience8;20,000 RL decisions per method.
**Clock:** stop learner work at10.5hours from first cell, finish reports by11.5hours, leaving30minutes before Kaggle's observed12hour kill.
Setup/model loading count against that clock. A partial checkpoint is reported as partial, not convergence.

**Real scope:** corrected CrisisMMD silver-priority D study. General multi-domain urgency and independent mission-E training remain blocked by missing independent labels.
The ordinary image pool is not relabeled Low. Teacher controls, actual swap consistency, strict label support and reward-validation gates are mandatory.
Qwen2.5-VL-7B at immutable revisioncc594898137f460bfe9f0759e9844b3ce807cfb5 uses NF4; no Qwen fine-tuning.
SQ-GAN is frozen, digital-byte source coding. Scheduler learning uses synthetic unit service; codec timing is separate.

## Required private inputs
1. Mini_dataset_6000.tar (SHA2560f8c6186b09dcf6ab87911df642612919377157bb1585082bed1c93b1506fe2c).
2. Scheduler_shared_cache_20261102.tar (SHA2568bbea66e64e64817d03e23e6b5332f548771a54d425a2a698d75f62a7a5abb44). It supplies identical frozen embeddings/mission vectors and SQ-GAN reference source.
3. WSL only:attach the published SQ-GAN teammate-2 output (existing semantic-notebook-2). No codec asset/training is required by Windows RL.

The Windows account currently cannot read the plus account's private cache. Attach private copies under Windows; do not make inputs public.
Neither notebook waits for the other. The Windows Deep Sets fit is an initialization prerequisite/frozen PPO comparator; do not count it as another independent WSL training seed.

Both notebooks profile exported policies on CPU:queue sizes4/16/64/256, selection milliseconds, parameter/weight size and supported-operator FLOPs. These FLOPs exclude some operations and are not a total compute claim. WSL also runs the baseline UE-rate study when training-window time remains.

## 1. Session, budget and recovery
Set RESUME_INPUT only to a complete prior run folder. Source/observations/schedules must match; otherwise start a new run. No files are deleted.

In [ ]:
SESSION_START=__import__('time').monotonic()
import os,sys,json,time,platform,hashlib,base64,zipfile,tarfile,io,shutil,datetime,contextlib,traceback
from pathlib import Path
os.environ.update({name:'1' for name in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS')})
PROFILE='wsl'
RESUME_INPUT=''  # Optional path to prior RUN folder attached read-only under /kaggle/input.
TRAINING_STOP=SESSION_START+10.5*3600
FINAL_STOP=SESSION_START+11.5*3600
SOURCE_SHA='76aa72a7d6dec6495f0eae941fb82d99e87bacbef582b779ff7c57fc28d5d34d'
WORK=Path('/kaggle/working')
if RESUME_INPUT:
    previous=Path(RESUME_INPUT)
    prior=json.loads((previous/'run_config.json').read_text())
    if prior['profile']!=PROFILE or prior['source_snapshot_sha256']!=SOURCE_SHA:
        raise ValueError('Resume requires the same profile and immutable source snapshot')
    RUN=(WORK/prior['relative_run']).resolve()
    if not RUN.is_relative_to(WORK/'experiments/scheduler_pair/runs') or RUN.exists():
        raise ValueError('Resume destination must be a new, verified working run path')
    shutil.copytree(previous,RUN)
else:
    RUN=WORK/'experiments/scheduler_pair/runs'/(PROFILE+'-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    RUN.mkdir(parents=True,exist_ok=False)
SESSION=RUN/'sessions'/datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SESSION.mkdir(parents=True,exist_ok=False)
(SESSION/'logs').mkdir()
CELL_RECORDS=[]
class Tee:
    def __init__(self,console,file):self.console,self.file=console,file
    def write(self,text):self.console.write(text);self.file.write(text);self.file.flush();return len(text)
    def flush(self):self.console.flush();self.file.flush()
    def isatty(self):return False
def persist_cells():
    (SESSION/'cells.json').write_text(json.dumps(CELL_RECORDS,indent=2))
    exported=[]
    for r in CELL_RECORDS:
        outputs=[]
        log=Path(r['log'])
        if log.exists():outputs.append({'output_type':'stream','name':'stdout','text':log.read_text(errors='replace').splitlines(keepends=True)})
        exported.append({'id':r['stage'],'cell_type':'code','metadata':{'stage':r['stage'],'status':r['status']},'source':r['source'].splitlines(keepends=True),'execution_count':r['execution_count'],'outputs':outputs})
    (SESSION/'executed_notebook.ipynb').write_text(json.dumps({'nbformat':4,'nbformat_minor':5,'metadata':{'capture':'cell source/stdout/stderr and errors; rich tables/figures saved separately; platform notebook is authoritative rich capture'},'cells':exported},indent=1))
@contextlib.contextmanager
def stage(name):
    shell=get_ipython()
    rec={'stage':name,'execution_count':shell.execution_count,'source':shell.history_manager.input_hist_raw[-1],
         'start_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'status':'running','log':str(SESSION/'logs'/(name+'.log'))}
    CELL_RECORDS.append(rec);persist_cells();started=time.monotonic()
    with Path(rec['log']).open('w',buffering=1) as log,contextlib.redirect_stdout(Tee(sys.stdout,log)),contextlib.redirect_stderr(Tee(sys.stderr,log)):
        print('START',name,'run',RUN,flush=True)
        try:
            yield
        except BaseException:
            rec['status']='failed';traceback.print_exc();raise
        else:rec['status']='complete'
        finally:
            rec['seconds']=time.monotonic()-started
            print('END',name,rec['status'],'seconds',round(rec['seconds'],2),flush=True)
            persist_cells()
with stage('01_session'):
    print({'profile':PROFILE,'interpreter':sys.executable,'python':sys.version,'platform':platform.platform(),
           'training_stop_hours':10.5,'finalization_stop_hours':11.5,'platform_limit_hours':12,
           'host_environment':'managed Kaggle, not a transferred local venv'},flush=True)

## 2. Frozen source snapshot
Both notebooks contain the same canonical source capsule. This is a disposable execution copy, not a second development codebase.

In [ ]:
with stage('02_pinned_source'):
    payload=base64.b64decode('...')
    if hashlib.sha256(payload).hexdigest()!=SOURCE_SHA:raise ValueError('source capsule mismatch')
    (RUN/'source_snapshot.zip').write_bytes(payload)
    SOURCE=RUN/'source';SOURCE.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        for member in z.infolist():
            if not (SOURCE/member.filename).resolve().is_relative_to(SOURCE):raise ValueError('unsafe source member')
        z.extractall(SOURCE)
    source_hashes={'Codec/catalog.py': '5dc051c82f70310967b29f46c8a519aea43630f8e6922e71615b332fb781ef04', 'Codec/config/scheduler_codec.json': '32b4d869860a7358b141c470176727efceb318a608299884000907a3ec5578ee', 'Codec/integration/sqgan_payload.py': 'd7452557bbea8ce2b0a163cdf720c845827a96cd93d49135b267d846021ea693', 'Evaluation/Artifacts/__init__.py': '4832c9bf7f00dbb097e6ab74647a112d7fa94c2247ae6c0946eb2dc9fc75f839', 'Evaluation/Artifacts/evaluation_artifact.py': '73a55dcd496fa18c97256e955b35f067f68ca2d0c9aad53788d0147bdbe0bc88', 'Evaluation/Artifacts/image_artifact.py': '35e90bbfb7cc62f9df800c4317b16d9598525cdf14b5b8b749ad7e3f58b320e6', 'Evaluation/Artifacts/knowledge_artifact.py': 'd535bda2ff082ac6241810be68856481ae51f6e7839fba7c149a6e03e9489acf', 'Evaluation/Artifacts/latent_artifact.py': '5bfae72383e1390489e87cafef6cecf061b0922a51c26250e5aaf6ecdae1920f', 'Evaluation/Artifacts/observation_artifact.py': 'f2687423a216817a02e387e6fae3f3282d1a506e53395ce028d8d191051a8199', 'Evaluation/Artifacts/scheduling_artifact.py': 'f7fce2af962ed91ace0fb3a2a4191f89e2f3a6bfc08c88d20615bc92e4f2ff72', 'Evaluation/Artifacts/semantic_packet_artifact.py': '4fd054101ba2e1cf83738aefa294ca780a8a19cab253fa91fcef8bf603ee8886', 'Evaluation/Artifacts/semantic_params_artifact.py': 'eecb6f56f2dbe24f478bf465408c33319f9b1f0ed8afca30ca74f2d6295975f5', 'Evaluation/Artifacts/transmission_artifact.py': '74d0089a4a4ded239593ef742bb96b7bac956ece6dbd0b02a0f40a8cab4d546e', 'Evaluation/Channels/__init__.py': '495c2fba2d8e8f84da06ef737bc49848bc104c419fae9f478a69f3f63708c949', 'Evaluation/Channels/awgn.py': '0a90a18e8e24a8377399194176e946ecd03fdfe115ec3c927bc732e64da84703', 'Evaluation/Channels/channel_factory.py': '76c16f298d007db7a0d22076cf5104876c9e2b499b406735ed12d8e2a1f4cc8c', 'Evaluation/Channels/channel_utils.py': 'f32a8771ff68d1d74c762efbba51b2ed9b757eac769e75d27085639f33c2501b', 'Evaluation/Channels/qam.py': '6b82cb859c7eed508fedf8b8f84545d2e9f63aca5fbae75d0351dbecd9c4483e', 'Evaluation/Channels/qpsk.py': 'c367ec9e9cd9c526447728b3dda19d5f09d70137a5d9f820e62713fc78073fa3', 'Evaluation/Channels/rayleigh.py': '0850a2563679ba7fb7799d5ec868eeed40629d829e6a48bc1aa89a13096a630d', 'Evaluation/Channels/rician.py': 'f9108f182f4937688277a274f3f8d5f457321e66c983b010732f9f3174d6fd55', 'Evaluation/Interfaces/__init__.py': '7d45ea35f10b8879c27b5077a759cafaebbc8494723e81b73f631f7680d0d358', 'Evaluation/Interfaces/channel_interface.py': 'b36a0b4d7ad2aba48c65dd97607b0768d9b7cb5776ebd3a87cd1d3baf30ee03e', 'Evaluation/Interfaces/codebook_interface.py': 'cac77059b42e5059f27bd3eed13ee862b546a20474b243a357311ff27f60e29e', 'Evaluation/Interfaces/deployment_interface.py': 'b2fee5db1f1948aff1d706c9affecc497aed0a30657c74f5f6b03becbe7d331e', 'Evaluation/Interfaces/encdec_interface.py': 'ce2e9c44fcb61d3d10ff5c4ed661ae492540a4eb613aedb203b1fa47cc0cf41e', 'Evaluation/Interfaces/evaluator_interface.py': 'bc763b2be6f2dd607a2f3380f0fe939874e7ed50c0f2149256d6757ae2f35a87', 'Evaluation/Interfaces/knowledge_interface.py': '87d836031fc68feb4340dd67d084259ff7c810d1b53aa0bd2aa895e8e7f691e2', 'Evaluation/Interfaces/latent_interface.py': 'a5a242bb1ce70a7cfd5c6a43fe55ea8a7d7bcecda6822c9df9d796c646a46886', 'Evaluation/Interfaces/param_extractor_interface.py': '3f6d29f21fc93a4a6ed542eaf50e4e8882d6e3fb18e477c4d0196dd4af07d5de', 'Evaluation/Interfaces/scheduler_interface.py': '7509f21df8b8d78c1f91bc8e5302e79eef60b33eb3f696c441ddb0fa058df9cf', 'Evaluation/Registries/__init__.py': '6e2a679f77e01be00064e3adc19065388b12914a2ee223d234fe3e86232f4f55', 'Evaluation/Registries/base_registry.py': '841decc8a7d739f230e7b6e88b78457a5c8b7bc530419a0493600f2f51f6d9e2', 'Evaluation/Registries/channel_registry.py': 'dc9cf8fca4bb97d5b5223fa68e3405437e9ee7bb38d373719a50ee23399f35c6', 'Evaluation/Registries/codebook_registry.py': '99392b0fc572c49f7bce54ad8b90f18df6e4d92104e69b73220d174f18060e4d', 'Evaluation/Registries/encdec_registry.py': '0e41d4dfd9b19f793e798275a87528c4fd9539e38cff451fc58ebcfb427d71c1', 'Evaluation/Registries/evaluator_registry.py': '4ed290db033642a09a76896f74304973c1a4639e2749e36b4c8e3ed7647ba638', 'Evaluation/Registries/knowledge_registry.py': 'ca2d5f2d7397594a5654c2dd73874ffca6274433f64a53cf8543e3d60dc53108', 'Evaluation/Registries/parameter_codebook_registry.py': '37eb7be7ceb797270bc7eddcafbe43e95726e46602bc2cb975c5f150a7b470a4', 'Evaluation/Registries/scheduler_registry.py': 'db8ea9b8d994ef26c89cb906c910cb3ccf3c59d487f8ccc9b06feb870db6dbfd', 'Evaluation/Registries/transformer_registry.py': 'f83df13a9439319adfb3b4b9b5591dfdc2a4b22b655f3fc37e399fc8f4a8ca3f', 'Evaluation/compact_scene_transport.py': '15a7f1e4aac1435761de5ee53f8cc245c223fb057d281f588928d999fb95caf9', 'Evaluation/errors.py': '64c4bdaa708c63f66cfd075ff2c2cbaec9aef53317d1f318607634939c32cb54', 'Evaluation/semantic_learning_env.py': '671affafec680012b27eb8ee7149dbc175086cc1a79e46a1abdcc9387858c2b6', 'Evaluation/ue_scenarios.py': 'd110494f4e019af37d640e47aa363be4be5c2cb7d5d0430ba21d794bcec777e9', 'Scheduler/semantic_learning_policy.py': '04af6847f5e8e462c31397e68d245059c50d3c87dca12801acb101752871bd58', 'Scheduler/tests/test_scene_learning_system.py': '89bd76fef7096b142dc482bbf374e72b01caa347c5edd8c4e87b1f7bdc2c550a', 'Scheduler/training/preference_reward.py': '307001447a75c561f1ec95e29e2088f78db7e5fadf1316d5284e12d9b85ee166', 'Scheduler/training/semantic_learning_models.py': 'f9aadee3371393b8f01b80ebd3d881e749e1fbbfd7b20f847fcb808d88e656a8', 'Scheduler/training/semantic_learning_rl.py': 'a0e61e89dcac7e27e172d551d0b404fdcc619e12099fbb856263627f9b3d8d0b', 'Scheduler/training/stream_suite/__init__.py': '1e3c7b5c71bc301c197ca2d4bfb7f9eea7ea8af9c6a0f0bb28cac99614e5a289', 'Scheduler/training/stream_suite/check_waiting_spec.py': 'c2a8eea5b8ca3d99c246bf7ecff1efefd5c4a34a8cfe279b2322e958725d2a7d', 'Scheduler/training/stream_suite/complete_vectors.py': 'd3a33f2aa3b2a89b14bd38c36e6689c09345a961bf34d5ba5e59f9174117b0dc', 'Scheduler/training/stream_suite/gpu_repair_validation.py': '3e0ac42a8b08f8ca0f76e2024bb689062f7dd06cf40e2107ccbc24435eaef1ae', 'Scheduler/training/stream_suite/gpu_teacher_format_check.py': 'db93d44626a6ae5221885b260f73b7cc525d395969b5a367cca303ffc52cd62c', 'Scheduler/training/stream_suite/input_preflight.py': 'cbdc1e269d861339143a57f04d741310ae46aceff0f33c76d7eb73134d5eca64', 'Scheduler/training/stream_suite/mini_cache_preflight.py': '0f3d08ef260b8912fa00a08cceb4bb63de57f5262973c061cf9982a1e67d0973', 'Scheduler/training/stream_suite/mini_kaggle_run.py': '2bac257413ccb0b23b168fadd19734b65c26d04fd05633a0c0254fe001f840d0', 'Scheduler/training/stream_suite/preflight_device.py': 'f2e72668baa4d046c0d5671548aa932c742afd615a0be5e7552c1b7176863777', 'Scheduler/training/stream_suite/prepare_teacher.py': '1712b6005f22bf35f59239ed48567e4388db23df88f44a37ecda5dfcd9367b34', 'Scheduler/training/stream_suite/profile_exports.py': '62ce55b3a65c6cdbef80ee341d8573af8c84e3fd138518713089f610ba48c2c3', 'Scheduler/training/stream_suite/qwen_teacher.py': '9cb24be8ab4a1bd76c2761c16820da3d491c8e8556aa7fe09eb8db8a7595a820', 'Scheduler/training/stream_suite/replay_external.py': '83b4ec0b345fe9b4a8cbeb6d8c4a9656891920b677da855a03b9b49f529cd5db', 'Scheduler/training/stream_suite/run_full_suite.py': '0af995ebbbda2e09fdcd611268d775a710081bdb2017a20d35fcc09da66e2565', 'Scheduler/training/stream_suite/run_guarded.py': '2fff0299b1c2b3185cfd7da019b3ebe60db4ec97c31671631acefd9dbea19c81', 'Scheduler/training/stream_suite/runtime_bootstrap.py': '4bec9f0489615811397c311766e94421bdca28839425e4c0a28a3ac0ea018775', 'Scheduler/training/stream_suite/semantic_suite_data.py': 'c878e337ff524b1d2065fe74ed551b377fca7464250828a2190b0afc6cb8f8c9', 'Scheduler/training/stream_suite/teacher_output.py': '5597e97417c30e1f3189cb51b0c99a25855e29fbae8a05f8d6a4ce21877892a7', 'Scheduler/training/stream_suite/teacher_panels.py': 'a13f66d091cdaed1c153d3b230c23ed3084ac6bfdef154b11e3505146aa65b51', 'Scheduler/training/stream_suite/teacher_pipeline.py': 'c3d09eac95b645efd33567a72e7d6e0cc7ebac3ca880f54470bca191cba92789', 'Scheduler/training/stream_suite/train_feedback_rl.py': 'a73dee929b42f0f9fc3bfc9f99ed043005747b3e011dc73e199eec77373cccfe', 'Scheduler/training/stream_suite/train_real_suite.py': '2e69e1749a2601c1323d8ee427901d7f6bdfa036c9ccde0529a116dfdd4bd701', 'Scheduler/training/stream_suite/train_teacher_rankers.py': '92f2e07e5fc1490bb6fa397b58435cfe42802b1bf640022c9740f932ca4a2012', 'Scheduler/training/stream_suite/ue_rate_study.py': 'ded4a8b6473f334d0bbffa1ebb972fc00fb248c7c6ea1b9828027155f3fb4ecf', 'Scheduler/training/stream_suite/validate_system.py': '622a2b09c8c54004197dfbbb045489078bc4a26c8bdc9045e423337488a9a1da', 'Scheduler/training/study/__init__.py': '1e3c7b5c71bc301c197ca2d4bfb7f9eea7ea8af9c6a0f0bb28cac99614e5a289', 'Scheduler/training/study/build_paired_notebooks.py': 'cdbbbfdc7d98e67b922eda3f5cb9d1f0e73ee1ac4d7048eb1d47ce073cab73d9', 'Scheduler/training/study/build_teacher_recovery_notebook.py': '5445bef8d7edf94638fded652dfe587ee36662ff0e6ec59cf8da3122f6ede717', 'Scheduler/training/study/master_suite.py': '49f38e48e61b25837eeeec73889e09e10ad8c178d233240b948c23dda2948c66', 'Scheduler/training/study/merge_paired_runs.py': 'b6d5fa12a19df3752e48017b3ec2ed0417e4ed01e9daad01657dbb546ecad704', 'Scheduler/training/study/paired_runtime.py': '4f757d9b371a168fe706a85b515a2149f9b6e48988b672f6159ff24f141d2621', 'Scheduler/training/study/requirements-kaggle.txt': '03dd1e9fb3b511ef1e61a24f80e9d42a7ae5c7174a8589e3980c481aaf8a8b0a', 'Scheduler/training/study/summarize_saved.py': 'ac2bb65ee647aefa66843174f10fe73b3997a0803075c749035e45ce0826a66c', 'Scheduler/training/study/teacher_recovery.py': 'e5a70bd288dd141cf51cf15085d9365b9a25849e86551ff3799f1f06d768921a', 'Test/tests/conftest.py': 'd3612a8b73e754f7b3790d7af57365bdb8ff3710fa11c46708268dfc265f1d64', 'Test/tests/test_channel_mapping.py': 'aaebb5ec7499d941fc168ea193e2f0cdb8b228eff1499a484397f30506728411', 'Test/tests/test_compact_scene_transport.py': 'c13149ed9f7c0e33cf3c39a40a1b6c7cae6e11d524c014998668a84cd50fcc70', 'experiments/master_study/master_suite.py': 'cd1ef6d49dcd0fe6100fcaab5be259d28656c9766d3b84ccc359e7e55fd7f73a', 'experiments/master_study/summarize_saved.py': '2a269f87341528b9ba0edd7a27027c5eb0a160ee080a57a197d036372c238bd9', 'experiments/master_study/tests/test_controller.py': 'dae23671f7556836a65b88b7da61dbff7aadec857f3f464b44575522cf1effe2', 'experiments/master_study/tests/test_paired_notebooks.py': 'f8107f71aa6ac4986cc397b58af67d31885b8179daf72b008ad61e7001d68dc5', 'experiments/master_study/tests/test_scientific_repairs.py': 'e4ec0d531e5e69b84c47813fada66e740df75fa54a0f38b83d4a5e4a2c0fd9ce', 'experiments/stream_scheduler/check_waiting_spec.py': '449e9e6049ad4e6097b1f868316b150b57f88e25bdc733a85415d49404894fc6', 'experiments/stream_scheduler/complete_vectors.py': 'ece7d82ae8f988e2f0e45bb516254ae7e9c26986e3247073de7658a36563d791', 'experiments/stream_scheduler/gpu_repair_validation.py': '3ca57641ca08ad6c60082045a111f4c59a68a8b9c270995f78b1e0672f8d885b', 'experiments/stream_scheduler/gpu_teacher_format_check.py': '3756a45b7716ef608f91df1339e9cf03e134d0e9da20d795190efb487bedda25', 'experiments/stream_scheduler/input_preflight.py': 'edf650812d47657fbb3ba888e70ae8f6ec2ad6efe9697dfa87fd947713c3b572', 'experiments/stream_scheduler/mini_cache_preflight.py': 'c4c5f55139f19d8bf341469c5129b4f37988ba2ae02ae8d500fdcdccace50084', 'experiments/stream_scheduler/mini_kaggle_run.py': '3a95461d805e3fbbb92149487c3e5997158a1b3dd34cdf4d8cc9564499aced26', 'experiments/stream_scheduler/preflight_device.py': '658303224b3195ea9750ec62e5070beb49ee05dd928fb5490f2afa70bbed34bd', 'experiments/stream_scheduler/prepare_teacher.py': '46a79d290dff3f99dd98d56c446d8b8268a46b5eb63a240d0f400709a0b63196', 'experiments/stream_scheduler/profile_exports.py': 'cf20993f50e2880b5401fcd5a4e83c5c11ca3ca61c77d059a1305c639b619896', 'experiments/stream_scheduler/qwen_teacher.py': '74ecc63de9810c055d39b593b017357e81054091b4aff74f3de987b163879d23', 'experiments/stream_scheduler/replay_external.py': '07ee3337d89238b19b3555ce9b4c68a6748c01c62ebda32f086a8689e8ca547d', 'experiments/stream_scheduler/requirements-teacher.txt': '55aff5609e56a25441cb8ab029691d255f9b3601cd5f5694e82af0992bbba6cf', 'experiments/stream_scheduler/run_full_suite.py': '6dfa7a8b7147ca4b75f2e67243e388fcb4276079fd98e7e90e00c42c2827ce7b', 'experiments/stream_scheduler/run_guarded.py': 'a4d5260e764097868ff9e8f48ee704f932ba960e383782e1c88a08ef2871fe56', 'experiments/stream_scheduler/runtime_bootstrap.py': '6414bbd286aaa2827fe8f5aa2926ca79ff9e59329e185914b4ddf8dca3fe559c', 'experiments/stream_scheduler/semantic_suite_data.py': 'c0132f8f514499eb9735fee1f40533ca957e661f0fc8c1e1f5ceba8f00342878', 'experiments/stream_scheduler/teacher_output.py': '153de1d492824fd237f19f8b33a0a4d6590588c75d6f01560c39e1485b542fdd', 'experiments/stream_scheduler/teacher_pipeline.py': '09251e91d1c5580a3b2c046cc2bcdc2865b173260885ee08f9421fc2bfc8088e', 'experiments/stream_scheduler/train_feedback_rl.py': '26df3dd7c15794ae219c2d1035111aee5706e3ef198c8823cfcab1137909e9dd', 'experiments/stream_scheduler/train_real_suite.py': '1eb371e10dbd15d61a090733af810aeb90ed4eef9256df04eff5bcaad2aa6029', 'experiments/stream_scheduler/train_teacher_rankers.py': '20ec90423158437ab3e18d90ded99ed6f9389b7f302fb9c181fc5c53f3ff3ad7', 'experiments/stream_scheduler/ue_rate_study.py': 'cbba106ac1f51eca140e007bcd2ee1dfa86db1ee042490c76b6e8412b1ae7c04', 'parameters/__init__.py': 'b3fb18ae6be9520a0ba82e9e41d8c5037aa685a4bb4dd8cb61d564cbb0a7de76', 'parameters/annotation_pilot.py': 'dce2f0b70dca949e9e44bb287c2d113c847012b0200011af1e9d515750793f56', 'parameters/annotation_review.py': 'ef0c808f897fa212fa461e92099dc78cb7288080c2ca2826ef4d689c8661f20d', 'parameters/approach_grid.py': '6ff61cfa3df0cec5cf8c91a0b2aec28e989c2fb055f9057354ee8c4a3143447c', 'parameters/artifacts.py': 'cd082fd07edca71cde4b9cdabef2749f744254aa96022add98da8bf82347d0ef', 'parameters/audit_trecis.py': '12ae6ec8d499c2d961606533194f347c0ea224e4bdd54f864e0b23d5245b8e84', 'parameters/build_annotation_notebook.py': '852e30c4e3110bd2baaec8d2b0a7829bb1108dbedb98952d80513ac07175e3f5', 'parameters/build_campaign.py': 'f40c80b4f0f4d367509b04c48fa91b4e6263ff3c186911be6a9bb8e34c8b9aca', 'parameters/build_grid_notebook.py': '1337ebbe082a8b8513dc87634d4b9c88ca745b2225a8626c78f5e552ed26ee25', 'parameters/build_journal_figure.py': '74993d589e5ed996804a1906e8e7f24b9466d8d8212c73c50b54503af7c70980', 'parameters/build_review_only_notebook.py': '73ab8e41c2e1e7531db086305aefef125e10e8dea7f49bcb0bc804fe127ea196', 'parameters/build_reviewer_followup_notebook.py': 'db2dc2ed4de520b283458b785e0b8a4e9479e8a2608a5833ab054cdea6dbe114', 'parameters/build_scene_review_notebook.py': 'c93ae38afc6ecabfaf5c3b1ed4cc57e3c846ac53ae4b294fa12809dd443b6c58', 'parameters/build_scene_schema.py': '7116354ee03d8885e69ecba17f6ac4c8df3723831fa60335d0373edfad3a985b', 'parameters/build_semantic_validation_notebook.py': 'b12ab583c706bc6e2c95159b560bf03f45a001dc04a49032cbba016262e42204', 'parameters/build_spi_final_notebook.py': '59afbe6f3794552ba513ac682a2629ce3b8deb25bfda5fd1db6905a100229ad1', 'parameters/build_spi_notebook.py': '70d093c8c23ca25f027fb61f515446b40704a611cce7eb81ce82e964764b2001', 'parameters/build_spi_pipeline_notebook.py': '0271f70bdc62e0832643885758d53b47e6d620b5a24068c686e0dc6e7b4b133d', 'parameters/build_walkthrough.py': 'df468805017408f79af8849ca8bc4376bd13d4afceb7b83171f1af7e8c597153', 'parameters/calibration.py': 'b294815dab5744b129b2803e1ab7c3106964777289d7e0f84d52e7c0de73e821', 'parameters/campaign_watch.py': 'addd3c37ec47a4d30e6d6f9186b7bb292419dc435bdbcde28acba851df0633d7', 'parameters/claim_repair_pilot.py': '9fa181daf299337c3c2762b8ff7a793019db575524fe0ffeaaed678dfb04d1f6', 'parameters/cli.py': 'e872f535ef465f3893bad380142c5b95520f236dc468e429648b24ba66763da2', 'parameters/cloud_data.py': '93cf9b4fe359ba29032b86deffa4aeecd67f2bc102c3e92964d4c2136d9b42c4', 'parameters/cloud_extract.py': 'f207448a33be9b303dad37b99519003a256061f8fe66e963361b31e7aa429a13', 'parameters/cloud_model_grid.py': 'd2c1732cf7d8cb30f2c63ff29310bba5e681798ecd1839514f056170224a1617', 'parameters/cloud_notebook.py': '4bed890b2e87e275c25445e30223374e6723676238255c26ba9f306401ad3469', 'parameters/cloud_study.py': '07ded64191a3736c970a2b7ded5b1dd7bb14db55bf61532d21000ee36ea22175', 'parameters/compare_spi_prompts.py': '4703045c038437e2adc2fb6097196fa33013bc728c721cdbd2cf300bca54b8d7', 'parameters/contracts.py': '3dede1c7168c0f9e0aee673a79c4e1489a61e4e0eaa1e2a14622e45d2ccc3e35', 'parameters/data.py': 'a1715bb04bd73580eb8e50f08d316aca2835d37ce749db56063cedb91f779800', 'parameters/dataset_pipeline.py': '45de4609bd167a7e0ae86ac800078ec1f6973e56be13bdcdff2875a87312ab15', 'parameters/evaluation.py': 'ab0195acabc3d7e3835714c669c56a163c7f7f04dc43d32d7318ea4abdc08cea', 'parameters/export_handoff_evidence.py': '8af849ad024ec42345626ade8e1928f95e88aaea5048ccea835113f7e8dd6d1a', 'parameters/exposure.py': 'e0fdb9de7b70fcfb2a4822a190fbd0d763149206da61c15b7d879b66d007db09', 'parameters/extend_spi_inputs.py': 'afa2c047a9156e964e0b6088e15951816d7ee06f5ab2fcd14804a86a46c4ed50', 'parameters/extraction_profile.py': 'b3b027e3b0d8295d143f32a874468d105aea83a25a80df716c918b77e37dd1ed', 'parameters/extractor.py': 'eabb61036c8219bf32a105f8bbccb4646a87c0f7f892a21bd8908b034fd1ffc6', 'parameters/gemma_spi_reviewer.py': '37af2bedd36bb0c4c32128f953f3834e271a48550f192a6213a88f368c9fa409', 'parameters/grid_inference.py': '1228a43fe73919f2609b07f6858f715f604326244685921512ec1ff09d21484a', 'parameters/grounding_scene.py': 'b8d70b5eef4d52cbaf9abacc77983bcaf2623ee36d621deb0140b8edcc5f2439', 'parameters/handoff_tools.py': '3269f6d1ea593809694bdbe0b61a22281acd9c6e86f8c072ab8fad7d3c4815c6', 'parameters/image_claims.py': '53c5643e3e907aec7e3a69fffb78284afca7916ce5815ef07635e7cd63fe912b', 'parameters/internvl_spi_reviewer.py': 'b028ef0ef0c22ef30208278447da427b268f3ce694decf616c9262c721b453cf', 'parameters/learning.py': '23fbbdec332915dec92ac7908fe39955a888681f8e50f8c7a4e70d4f7f7d4e53', 'parameters/minimal_parameters.py': 'ca82742e316c3015a5258abb9ceaa4790dbebbd94803bc978adad14d1f978fdc', 'parameters/minimal_vision.py': '9bf4a63e311f5f891929e0c194d34f081d9921945599fc86f726e093ced77542', 'parameters/model_settings.py': '4400bd5a4dad3592f523271e41a1086493c729461ee88a787d86ec99f1c68f6a', 'parameters/native_vlm_candidates.py': 'e69d1a07e0c701aec74faf4ec76a72cd6dd59369611e78065e0167cfbe1da0b0', 'parameters/official_panel.py': '4202ed6528485adfebe968b0abbfbae87866d678f20709eafef8cf3e63871d5b', 'parameters/parameter_cli.py': '4687341e9b5855005ae63182ceac12fae8a7d6af03eeaecc984e03c0e229daf8', 'parameters/parameter_experiments.py': 'a3fb0ee5d1b476e06b7c7f87550cb032a93e14c1288db727db5760d96ea9cf8c', 'parameters/parameter_vlm_review.py': 'c11a471ad3a5608110b7c3cae3d40243820871c55b466227aa131b313361218d', 'parameters/pilot_parameters.py': '5c795efa1ce4539afe8879a66fa0465b30536a6ea1d3cd055a1a11679979aa6b', 'parameters/pilot_producers.py': 'a333f2f2405d1742aff5a59366b32d5969b5e835fd872ed606fe9a0fc0e8f03b', 'parameters/preflight_reviewer_processor.py': 'd325451d7a0d25062e3bd33eb73fd1bc4c8faf732af7f8e9a2b7b7b81c426fcf', 'parameters/qwen35_reviewer.py': 'fd8f89976c7828c7274fcfcda0801176ee881f2bc5c891dd6b62fea6bebbbb52', 'parameters/qwen_reviewer.py': '594ca60fa7fc04f84c3ea04015b22e699ffd5a910134172ee3bc82dc66564905', 'parameters/readiness.py': '20938e6b955467e9e5f744b08b5cc3401fbf2453dee747f4d6ba4bc407cbef07', 'parameters/replay_pilot.py': 'a4f6ce4afa62c971a3765cc55343b165900530aba10a9a17461bbafcf5cadcf7', 'parameters/review_only.py': 'd49bd96f1ebc7f398839ee1d41029cc4f4e4d7c1c640a3ef776cbf30b887f388', 'parameters/reviewer_followup.py': '645a83dc3e944cc5b25bf58cfeb2a311f5f4fb7065fe0dfe9ceb000969324c39', 'parameters/reviewer_grid.py': '45d6fe89499eda3f993aa58e0d2bccdf43e1124d8f16fb88fbde24383feb2520', 'parameters/scene_arrays.py': 'd055954dbab2b3dbf814e5db3d1cde90381e9e9e264191e76e86ab95c92f0481', 'parameters/scene_json.py': '4bbeee761acfb5f85a1f84af5227d39450d19791ae785505ebdb52073a48a440', 'parameters/scene_json_replay.py': 'dd6feb6d4d1b64a422d07503cfe280774ae53b7da619b043d8b92f6c23ecf0cd', 'parameters/scene_measurements.py': 'cdb04313661f60ab13d6b1e4b15967865c1e3c00403606d7429b8c45b87edced', 'parameters/scene_review.py': '7f4e5d76e37e20d6244a84255e999daa1a6c96948662959ee100271251ab9630', 'parameters/scene_review_run.py': 'aadd82c458b736ce954a920d57334c783207a165fc87b81a9f944e6eee0993ec', 'parameters/scene_vocabulary.py': 'd2e75854b5180ae8127ada69bc3735d7f1c6ea7c00ffc159986bb2c830c32942', 'parameters/semantic_contract.py': 'ea8e4417ea31baa1a9eb632f413bcf233bd34e89cea404ff5826ee050e1607ac', 'parameters/semantic_panel.py': '05a5f2aecb22b0cbd1976bf99df619e684468286ff7f3b43f7cd458c3fde864a', 'parameters/semantic_validation_driver.py': 'd2123e9116f1a09cf6bb0009cb7575953c999291a85d369458634eb1a46e9c53', 'parameters/semantic_validation_run.py': 'ab979882e12fca39fbe633a71efa3ce7e56d9f47e922aa8341bb701358bd8894', 'parameters/serialization.py': '25b1a0f19bdc386c5f8cfa05ff0d871ae7c5b78ac3ff72ffe574de47ecfe0338', 'parameters/spatial_provider.py': '290713c277c9789f10176f053d320d11b967fd878c077e4bcd877a3ca6e3edd1', 'parameters/spi.py': '961f7214fcf33c3772d144b5f6accac780b3c591261099b80b0196fc40393f0a', 'parameters/spi_campaign.py': 'bd23ee5d59ef23d32a37e09315fd6d6c65caced1828d5fae21194a00b21699fa', 'parameters/spi_direct_priority.py': '98975d2ac0d2b1c42cf2dbdbccabf2749dcf22ec1866463a7afa7ebb1c4b75ae', 'parameters/spi_final_campaign.py': '9df76ddc50f9006073181d561915646a0fde30df2c066f6957ff10440224f342', 'parameters/spi_infer.py': 'adf3282f55092ffca442091eb73bc9d4930d48de3aca068ec09a9ae18416ad3e', 'parameters/spi_labels.py': '366360db6dfb0ce6298ee12511a9cf03bc0593c59b5b1c8c3fdf3ddef5427cbb', 'parameters/spi_model_selection.py': '08c2856b602a55f96afe294426dbd828239a6fd35ace08e918df4b93b1a21080', 'parameters/spi_pipeline.py': '898a30f1d288fb57ce9cca755d8b74058c2096f15f2c7e6b3ba60c8990b6683b', 'parameters/spi_protocol.py': '6ec916d9016e72a426e82e2ea6666e8507fc5ca6408fc62d8031865e1dad404e', 'parameters/spi_release.py': '3bf99c56307dd9edd368a5d83316a1e61e0f377e8a11d674ee40e7b6dab5c0cd', 'parameters/spi_teachers.py': 'c5cf34307afcc4d8a94130ef41452e4cdbe173f5caa908efe5568eee5ec033d7', 'parameters/study.py': '6653a4459c5f3378caa710f5a4f05b38b93fe4b8cbfda399bb5c76d03d07a885', 'parameters/ue_profiles.py': '91fda13f758bce8faafe190d20de1d5104ffc156e22e0dd992081c8cb2af5954', 'parameters/vision.py': '051b23acb96dbeb58bb435ff575c9a65289b6c023e5a6256e3c03e3b99edcbb0', 'project_compat.py': '8e97f72ece2a2b555d806bb7d2d56cca390a40138789e956e47ad3d78856f72d', 'src/Channels/__init__.py': '82573747007eb88ea68bce059369fea8f6b88776746dbe0088c1553877e8795a', 'src/Channels/channel_utils.py': '763dade3575cba87f9509428937018b1e6bd7cac6fecd268dd301124e5b0e4d4'}
    for name,expected in source_hashes.items():
        if hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()!=expected:raise ValueError('source mismatch:'+name)
    sys.path.insert(0,str(SOURCE))
    import importlib.util
    spec=importlib.util.spec_from_file_location('runtime_bootstrap',SOURCE/'Scheduler/training/stream_suite/runtime_bootstrap.py')
    bootstrap=importlib.util.module_from_spec(spec);spec.loader.exec_module(bootstrap);bootstrap.install(SOURCE)
    cfg={'profile':PROFILE,'relative_run':str(RUN.relative_to(WORK)),'source_snapshot_sha256':SOURCE_SHA,
         'source_hashes':source_hashes,'seed':1729,'epochs_max':50,'rl_decisions_max':20000}
    (RUN/'run_config.json').write_text(json.dumps(cfg,indent=2))
    print('Verified source files',len(source_hashes),flush=True)

## 3. Input identity
Fail before GPU work if either private input is absent or wrong. The same cache avoids producer differences across accounts.

In [ ]:
with stage('03_inputs_and_cache'):
    archives=[p for name in ('Mini_dataset_6000.tar','Mini_dataset_6000.tar.bin') for p in Path('/kaggle/input').rglob(name)]
    shared=[p for name in ('Scheduler_shared_cache_20261102.tar','Scheduler_shared_cache_20261102.tar.bin') for p in Path('/kaggle/input').rglob(name)]
    if len(archives)!=1 or len(shared)!=1:raise ValueError('Attach exactly one mini archive and one shared cache; found '+str((len(archives),len(shared))))
    ARCHIVE=archives[0]
    from Scheduler.training.study.paired_runtime import digest
    if digest(ARCHIVE)!='0f8c6186b09dcf6ab87911df642612919377157bb1585082bed1c93b1506fe2c':raise ValueError('image archive mismatch')
    if digest(shared[0])!='8bbea66e64e64817d03e23e6b5332f548771a54d425a2a698d75f62a7a5abb44':raise ValueError('shared cache mismatch')
    COMMON=RUN/'shared'
    if not (COMMON/'shared_manifest.json').exists():
        COMMON.mkdir(exist_ok=True)
        with tarfile.open(shared[0]) as tar:
            for item in tar.getmembers():
                if item.issym() or item.islnk() or not (COMMON/item.name).resolve().is_relative_to(COMMON):raise ValueError('unsafe cache member')
            tar.extractall(COMMON,filter='data')
    common_manifest=json.loads((COMMON/'shared_manifest.json').read_text())
    for name,expected in common_manifest['files'].items():
        if digest(COMMON/name)!=expected:raise ValueError('cache mismatch:'+name)
    VECTORS=COMMON/'vectors/vectors.npz'
    MISSIONS=COMMON/'missions/mission_vectors.npy'
    identity={'archive':str(ARCHIVE),'archive_sha256':digest(ARCHIVE),
        'shared_sha256':digest(shared[0]),'vectors_sha256':digest(VECTORS),'mission_vectors_sha256':digest(MISSIONS)}
    if RESUME_INPUT:
        previous_identity=json.loads((RUN/'input_manifest.json').read_text())
        if previous_identity!=identity:raise ValueError('Resume input identity/path mismatch')
    (RUN/'input_manifest.json').write_text(json.dumps(identity,indent=2))
    print('Image archive and frozen caches verified. No SPI/scene/CLIP extraction is rerun.',flush=True)

## 4. Managed environment and real hardware
Both require Kaggle T4×2 and internet for pinned model/config downloads. Package setup counts toward the session budget.

In [ ]:
with stage('04_environment'):
    import subprocess
    dependencies=[line.strip() for line in (SOURCE/'Scheduler/training/study/requirements-kaggle.txt').read_text().splitlines() if line.strip() and not line.startswith('#')]
    command=[sys.executable,'-m','pip','install',*dependencies]
    print('Install into active kernel:',command,flush=True)
    subprocess.run(command,check=True,timeout=600)
    import torch,numpy,lightgbm,transformers
    actual={'python':sys.version,'torch':torch.__version__,'numpy':numpy.__version__,'lightgbm':lightgbm.__version__,
            'transformers':transformers.__version__,'cuda':torch.cuda.is_available(),
            'devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    if RESUME_INPUT:
        previous_env=[p for p in RUN.glob('sessions/*/environment.json') if p.parent!=SESSION]
        if not previous_env:raise ValueError('Resume has no prior environment identity')
        prior_environment=json.loads(sorted(previous_env)[-1].read_text())
        for key in ('python','torch','numpy','lightgbm','transformers'):
            if prior_environment[key]!=actual[key]:raise ValueError('Resume environment differs:'+key+'; use an explicitly separate weights-only experiment instead')
    (SESSION/'environment.json').write_text(json.dumps(actual,indent=2));print(actual,flush=True)
    if torch.cuda.device_count()!=2:raise RuntimeError('This allocation requires two CUDA devices; refuse CPU fallback')
    for i in range(2):
        probe=torch.ones(1,device='cuda:'+str(i));assert probe.device.index==i
        torch.cuda.synchronize(i)
    del probe
    print('Managed Torch/CUDA retained; no core-framework replacement. Fresh kernel required if dependencies were previously imported.',flush=True)

## 5. Numerical/lifecycle tests and reference audit
Check both CUDA allocations, SQ01 digital-packet CRC, scientific repair regressions and controller execution. No scheduler real-data fitting occurs in this cell.

In [ ]:
with stage('05_contracts_and_reference_scope'):
    from Scheduler.training.stream_suite.input_preflight import check
    coverage=check(ARCHIVE,RUN/'reference_preflight',['crisismmd'])
    if coverage['status']=='blocked_references':raise RuntimeError('Even the scoped priority cohort is unavailable')
    print('Only CrisisMMD has D references. Ordinary-domain and independent mission-E claims remain blocked.',flush=True)
    command=[sys.executable,'-m','pytest','-q','-ra','-c','/dev/null','--rootdir',str(SOURCE),'--confcutdir',str(SOURCE),
             str(SOURCE/'experiments/master_study/tests/test_scientific_repairs.py'),
             str(SOURCE/'experiments/master_study/tests/test_controller.py')]
    subprocess.run(command,cwd=SOURCE,env={**os.environ,'PYTHONPATH':str(SOURCE)},check=True,timeout=600)
    spec=importlib.util.spec_from_file_location('sq01_packet_preflight',COMMON/'sqgan_reference/src/bitstream.py')
    packet_format=importlib.util.module_from_spec(spec);spec.loader.exec_module(packet_format)
    test_packet=packet_format.encode_packet([0,511],[0,1023],[1],[17])
    assert packet_format.decode_packet(test_packet)==[([0,511],[0,1023]),([1],[17])]
    corrupted=bytearray(test_packet);corrupted[-1]^=1
    try:packet_format.decode_packet(bytes(corrupted))
    except ValueError:pass
    else:raise AssertionError('SQ01 corruption was accepted')
    print('Packaged regression subset and SQ01 CRC checks passed. No analog channel is used.',flush=True)

## 6. SQ-GAN: one image, then five
Frozen checkpoint only. Record real encode/decode times and byte payloads; these do not replace synthetic service times in learning. Native256×512 preprocessing is recorded. This small check does not establish p95 latency or disaster semantic quality.

In [ ]:
import contextlib,tempfile,shutil
with stage('06_frozen_sqgan_payload_check'), contextlib.ExitStack() as cleanup:
    from Codec.integration.sqgan_payload import SQGANDigitalCodec
    candidates=[p.parent for p in Path('/kaggle/input').rglob('bundle.json')
                if p.parent.name=='20261028T130100Z-eea4f3aa' and p.parent.parent.name=='SQ-GAN']
    if not candidates:
        flat=list(Path('/kaggle/input').rglob('scheduler_sqgan_flat_manifest.json'))
        if len(flat)!=1:raise ValueError('Attach exactly one verified SQ-GAN bundle or flat package')
        mapping=json.loads(flat[0].read_text())
        if mapping.get('schema')!='scheduler_sqgan_flat_v1' or mapping.get('run_id')!='20261028T130100Z-eea4f3aa':
            raise ValueError('Unexpected flat SQ-GAN package identity')
        source=flat[0].parent
        manifest_path=(source/mapping['bundle_manifest']).resolve()
        if manifest_path.parent!=source.resolve() or digest(manifest_path)!='8b482d80a1adc7dde7a2725461a17346aca49d6628d4f78f88f1619df7448c21':
            raise ValueError('Original SQ-GAN bundle manifest mismatch')
        original=json.loads(manifest_path.read_text())
        required=sum(entry['bytes'] for entry in original['files'])
        if shutil.disk_usage('/kaggle/working').free<required+512*1024*1024:
            raise ValueError('Insufficient Kaggle disk for verified codec materialization')
        temporary=Path(cleanup.enter_context(tempfile.TemporaryDirectory(prefix='sqgan-input-',dir='/kaggle/working')))
        bundle=temporary/'SQ-GAN'/mapping['run_id'];bundle.mkdir(parents=True)
        shutil.copyfile(manifest_path,bundle/'bundle.json')
        for entry in original['files']:
            member=mapping['files'][entry['path']]
            if isinstance(member,dict):
                if member.get('bytes')!=entry['bytes'] or member.get('sha256')!=entry['sha256']:
                    raise ValueError('Flat entry disagrees with original bundle manifest')
                filename=member.get('file')
            else:
                filename=member
            if not isinstance(filename,str):raise ValueError('Flat entry filename must be a string')
            uploaded=(source/filename).resolve()
            destination=(bundle/entry['path']).resolve()
            if uploaded.parent!=source.resolve() or not destination.is_relative_to(bundle):
                raise ValueError('Unsafe flat SQ-GAN member')
            if uploaded.stat().st_size!=entry['bytes']:
                raise ValueError('Incomplete uploaded SQ-GAN member: '+entry['path'])
            destination.parent.mkdir(parents=True,exist_ok=True)
            shutil.copyfile(uploaded,destination)
            print('Materialized codec input',entry['path'],entry['bytes'],flush=True)
        candidates=[bundle]
    if len(candidates)!=1:raise ValueError('Attach the selected teammate2 SQ-GAN output; found '+str(len(candidates)))
    # Verify one image, then five more. Cache each complete packet once.
    import tarfile,numpy as np
    from PIL import Image,ImageOps
    codec=SQGANDigitalCodec(bundle=candidates[0],source_root=COMMON/'sqgan_reference',device='cuda:0')
    destination=RUN/'sqgan_payloads';destination.mkdir(exist_ok=True)
    records=[]
    with tarfile.open(ARCHIVE) as tar:
        index=json.load(tar.extractfile('Mini_dataset/portable_image_index.json'))
        for j,item in enumerate(index[:6]):
            with Image.open(tar.extractfile('Mini_dataset/'+item['relative_path'])) as opened:
                image=ImageOps.exif_transpose(opened).convert('RGB')
                width,height=image.size;factor=min(512/width,256/height)
                resized=image.resize((max(1,round(width*factor)),max(1,round(height*factor))))
                canvas=Image.new('RGB',(512,256),(127,127,127));offset=((512-resized.width)//2,(256-resized.height)//2);canvas.paste(resized,offset)
            x=torch.from_numpy(np.asarray(canvas).copy()).permute(2,0,1).float()[None].to('cuda:0')/255
            torch.cuda.synchronize();start=time.perf_counter();semantic=codec.semantic_map(x)
            torch.cuda.synchronize();teacher_seconds=time.perf_counter()-start
            start=time.perf_counter();packet=codec.encode(x,semantic);torch.cuda.synchronize();encode_seconds=time.perf_counter()-start
            path=destination/(str(j)+'.sq01');path.write_bytes(packet)
            start=time.perf_counter();decoded=codec.decode(packet);torch.cuda.synchronize();decode_seconds=time.perf_counter()-start
            Image.fromarray(np.round(decoded[0].detach().cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)).save(destination/(str(j)+'_reconstruction.png'))
            records.append({'image_id':item['image_id'],'payload_bytes':len(packet),'payload_sha256':digest(path),
                            'teacher_seconds':teacher_seconds,'encode_seconds':encode_seconds,'decode_seconds':decode_seconds,
                            'geometry':{'source_width':width,'source_height':height,'scale':factor,'offset':offset},
                            'checkpoint_sha256':codec.checkpoint_sha256})
            (destination/'manifest.json').write_text(json.dumps({'scope':'frozen source codec; not lossless/radio/final semantic validation','records':records},indent=2))
            print('SQGAN',j+1,'of6','bytes',len(packet),'decode_seconds',round(decode_seconds,4),flush=True)
    del codec,x,semantic,decoded
    import gc;gc.collect();torch.cuda.empty_cache()

## 7. Assigned training
GPU0:10 supervised D methods. GPU1:Qwen candidate/prefix/replay labels,7 P students,2 feedback RL methods, each gated.
Baseline policies are evaluated before learned methods. Checkpoints contain optimizer/progress/RNG state and are atomically replaced. A cutoff marks unfinished methods partial. VLM label failure blocks only dependent methods; no invented labels are substituted.

In [ ]:
with stage('07_training_and_conditional_labels'):
    from Scheduler.training.study.paired_runtime import PairRun
    config={'profile':PROFILE,'run':str(RUN),'archive':str(ARCHIVE),'vectors':str(VECTORS),'mission_vectors':str(MISSIONS),
            'training_stop_monotonic':TRAINING_STOP,'final_stop_monotonic':FINAL_STOP}
    (SESSION/'resolved_execution.json').write_text(json.dumps(config,indent=2))
    runner=PairRun(config)
    try:runner.execute()
    finally:
        runner.report()
        print('Training stage exited; consult per-method state, not only process exit.',flush=True)

## 8. Results and recovery
Development-only comparisons, class-delay/miss safeguards, raw status tables and plots. Do not rank feedback versus D from different traces. One seed is a trial, not stability evidence. Keep failures and blocked methods visible.

In [ ]:
with stage('08_tables_figures_and_recovery'):
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    for path in sorted((RUN/'reports').glob('*/delay_comparison.csv')):
        table=pd.read_csv(path)
        if table.empty:
            print('No shared-D rows for',path.parent.name,'; see its method/status report.',flush=True);continue
        print('Shared D validation:',path.parent.name,flush=True);display(table)
        table.to_html(SESSION/(path.parent.name+'_delay.html'),index=False)
        ax=table.set_index('method')['mean_wait_penalty'].plot.barh(figsize=(9,max(4,.3*len(table))))
        ax.set_xlabel('D normalized waiting penalty (lower better; synthetic time)')
        ax.set_title(path.parent.name+' — development/silver results')
        ax.figure.tight_layout();ax.figure.savefig(SESSION/(path.parent.name+'_delay.png'),dpi=150);plt.show();plt.close(ax.figure)
    status=json.loads((RUN/'RUN_STATUS.json').read_text())
    display(pd.DataFrame.from_dict(json.loads((RUN/'method_ledger.json').read_text()),orient='index'))
    print('RUN STATUS',status,flush=True)
    print('Runtime/partial-FLOPs profiles:',list((RUN/'profiles').glob('*.json')),flush=True)
    print('Important:Critical waits/misses are in critical_safeguards.json; aggregate D is not a Critical guarantee.',flush=True)
    print('Retrieve the ENTIRE run folder, including source/cache identity, session logs, raw teacher records, optimizer checkpoints and metrics.',flush=True)
    print('Resume:attach that run under /kaggle/input and set RESUME_INPUT. Preserve original input mount names and source snapshot.',flush=True)
    (SESSION/'artifact_index.json').write_text(json.dumps({str(p.relative_to(RUN)):{'bytes':p.stat().st_size}
          for p in RUN.rglob('*') if p.is_file()},indent=2))
    print('Incremental executed_notebook.ipynb is in',SESSION,'; platform executed notebook/rich output must also be retrieved.',flush=True)